In [2]:
words = open('names.txt', 'r').read().splitlines()

words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
import torch

N = torch.zeros((27*27, 27), dtype=torch.int32)  # 27*27 situation for taking two characters 

chars = ['.'] + sorted(set(''.join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

In [4]:
xs, ys = [], []

for w in words:
    chs = ['.'] + ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1] * 27 + stoi[ch2]
        ix2 = stoi[ch3]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27*27, 27), generator=g, requires_grad=True)

number of examples:  228146


In [9]:
import torch.nn.functional as F



# gradient descent
for i in range(50):
    # forward pass
    # xenc = F.one_hot(xs, num_classes=27*27).float()
    # logits = xenc @ W

    #replace one-hot
    logits = W[xs]
    
    # softmax
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    
    # loss
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()

    # print(loss.item())
    
    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

loss.item()

2.694852590560913

logits = W[xs] 同 可见 torch.index_select

# torch.index_select

```python
torch.index_select(input, dim, index, *, out=None) → Tensor
```

沿指定维度 `dim`，根据 `index` 中的索引从输入张量 `input` 中选择数据，并返回一个新张量。

返回张量与原始张量 `input` 的维度数量相同。其中：

- 第 `dim` 个维度的大小等于 `index` 的长度。
- 其他维度的大小与原始张量相同。

> **注意**
>
> 返回张量与原始张量不共享存储空间。
>
> 如果 `out` 的形状与预期形状不同，PyTorch 会自动将其调整为正确形状，并在必要时重新分配底层存储空间。

## 参数

- **input** (`Tensor`)：输入张量。
- **dim** (`int`)：执行索引选择的维度。
- **index** (`IntTensor` 或 `LongTensor`)：包含所选索引的一维张量。

## 关键字参数

- **out** (`Tensor`, 可选)：用于保存结果的输出张量。

## 示例

```python
>>> x = torch.randn(3, 4)
>>> x
tensor([[ 0.1427,  0.0231, -0.5414, -1.0009],
        [-0.4664,  0.2647, -0.1228, -1.1068],
        [-1.1734, -0.6571,  0.7230, -0.6004]])

>>> indices = torch.tensor([0, 2])

>>> torch.index_select(x, 0, indices)
tensor([[ 0.1427,  0.0231, -0.5414, -1.0009],
        [-1.1734, -0.6571,  0.7230, -0.6004]])

>>> torch.index_select(x, 1, indices)
tensor([[ 0.1427, -0.5414],
        [-0.4664, -0.1228],
        [-1.1734,  0.7230]])
```

在第一个调用中：

```python
torch.index_select(x, 0, indices)
```

`dim=0` 表示选择行，`indices=[0, 2]` 表示选择第 0 行和第 2 行。

在第二个调用中：

```python
torch.index_select(x, 1, indices)
```

`dim=1` 表示选择列，`indices=[0, 2]` 表示选择第 0 列和第 2 列。